# `go-huggingface` Demo

This demo shows how to download files and create tokenizers from HuggingFace models.

In [1]:
import (
    "github.com/janpfeifer/must"
    "github.com/gomlx/compute/support/humanize"
    "github.com/gomlx/go-huggingface/hub"
    "github.com/gomlx/go-huggingface/tokenizers"
)

---

## HuggingFace Hub: Downloading files

Download `tokenizer_config.json` and enumerate `tokenizer_class` for several models

In [24]:
var (
    // HuggingFace authentication token read from environment.
    // It can be created in https://huggingface.co
    // Some files may require it for downloading.
    hfAuthToken = os.Getenv("HF_TOKEN")

    // Model ids for testing.
    hfModelIDs = []string{
        "google/gemma-2-2b-it",
        "sentence-transformers/all-MiniLM-L6-v2",
        "protectai/deberta-v3-base-zeroshot-v1-onnx",
        "KnightsAnalytics/distilbert-base-uncased-finetuned-sst-2-english",
        "KnightsAnalytics/distilbert-NER",
        "KnightsAnalytics/all-MiniLM-L6-v2",
        "SamLowe/roberta-base-go_emotions-onnx",
    }
)

In [6]:
%%
for _, modelID := range hfModelIDs {
    fmt.Printf("\n%s:\n", modelID)
    repo := hub.New(modelID).WithAuth(hfAuthToken)
    for fileInfo, err := range repo.IterFileInfos() {
        if err != nil { panic(err) }
        fmt.Printf("\t%s - %s\n", fileInfo.Name, humanize.Bytes(fileInfo.Size))
    }
}


google/gemma-2-2b-it:
	.gitattributes - 1.5 KiB
	README.md - 28.4 KiB
	config.json - 838 B
	generation_config.json - 187 B
	model-00001-of-00002.safetensors - 4.6 GiB
	model-00002-of-00002.safetensors - 229.5 MiB
	model.safetensors.index.json - 23.7 KiB
	special_tokens_map.json - 636 B
	tokenizer.json - 16.7 MiB
	tokenizer.model - 4.0 MiB
	tokenizer_config.json - 45.9 KiB

sentence-transformers/all-MiniLM-L6-v2:
	.gitattributes - 1.2 KiB
	1_Pooling/config.json - 190 B
	README.md - 10.3 KiB
	config.json - 612 B
	config_sentence_transformers.json - 116 B
	data_config.json - 38.3 KiB
	model.safetensors - 86.7 MiB
	modules.json - 349 B
	onnx/model.onnx - 86.2 MiB
	onnx/model_O1.onnx - 86.2 MiB
	onnx/model_O2.onnx - 86.1 MiB
	onnx/model_O3.onnx - 86.1 MiB
	onnx/model_O4.onnx - 43.1 MiB
	onnx/model_qint8_arm64.onnx - 22.0 MiB
	onnx/model_qint8_avx512.onnx - 22.0 MiB
	onnx/model_qint8_avx512_vnni.onnx - 22.0 MiB
	onnx/model_quint8_avx2.onnx - 22.0 MiB
	openvino/openvino_model.bin - 86.1 MiB


In [7]:
%%
for _, modelID := range hfModelIDs {
    fmt.Printf("\n%s:\n", modelID)
    repo := hub.New(modelID).WithAuth(hfAuthToken)
    config := must.M1(tokenizers.GetConfig(repo))
    fmt.Printf("\ttokenizer_class=%s\n", config.TokenizerClass)
}


google/gemma-2-2b-it:
	tokenizer_class=GemmaTokenizer

sentence-transformers/all-MiniLM-L6-v2:
	tokenizer_class=BertTokenizer

protectai/deberta-v3-base-zeroshot-v1-onnx:
	tokenizer_class=DebertaV2Tokenizer

KnightsAnalytics/distilbert-base-uncased-finetuned-sst-2-english:
	tokenizer_class=DistilBertTokenizer

KnightsAnalytics/distilbert-NER:
	tokenizer_class=DistilBertTokenizer

KnightsAnalytics/all-MiniLM-L6-v2:
	tokenizer_class=BertTokenizer

SamLowe/roberta-base-go_emotions-onnx:
	tokenizer_class=RobertaTokenizer


---

## HuggingFace Tokenizers

### Go-only SentencePiece tokenizer:

In [8]:
var sentence = "The book is on the table."

%%
repo := hub.New("google/gemma-2-2b-it").WithAuth(hfAuthToken)
tokenizer := must.M1(tokenizers.New(repo))
tokens := tokenizer.Encode(sentence)
fmt.Printf("Sentence:\t%s\n", sentence)
fmt.Printf("Tokens:  \t%v\n", tokens)


Sentence:	The book is on the table.
Tokens:  	[2 651 2870 603 611 573 3037 235265]


In [11]:
%%
repo := hub.New("KnightsAnalytics/all-MiniLM-L6-v2").WithAuth(hfAuthToken)
tokenizer := must.M1(tokenizers.New(repo))
tokens := tokenizer.Encode(sentence)
fmt.Printf("Sentence:\t%s\n", sentence)
fmt.Printf("Tokens:  \t%v\n", tokens)

Sentence:	The book is on the table.
Tokens:  	[101 1996 2338 2003 2006 1996 2795 1012 102]


### Rust based [github.com/daulet/tokenizers](https://github.com/daulet/tokenizers) tokenizer

In the rare case where you want more performance, or if you find a tokenizer not yet supported by Go only (please, create an issue if that is the case), you can use the [github.com/daulet/tokenizers](https://github.com/daulet/tokenizers), which is based on a fast tokenizer written in Rust.

It requires installation of the built Rust library though, see [github.com/daulet/tokenizers](https://github.com/daulet/tokenizers) on how to install it, they provide prebuilt binaries.

> **Note**: `daulet/tokenizers` also provides a simple downloader, so `go-huggingface` is not strictly necessary -- if you don't want the extra dependency and only need the tokenizer, you don't need to use it. `go-huggingface` helps by allowing also downloading other files (models, datasets), and a shared cache across different projects and `huggingface-hub` (the python downloader library).

In [12]:
import dtok "github.com/daulet/tokenizers"

%%
modelID := "KnightsAnalytics/all-MiniLM-L6-v2"
repo := hub.New(modelID).WithAuth(hfAuthToken)
localFile := must.M1(repo.DownloadFile("tokenizer.json"))
tokenizer := must.M1(dtok.FromFile(localFile))
defer tokenizer.Close()
tokens, _ := tokenizer.Encode(sentence, true)

fmt.Printf("Sentence:\t%s\n", sentence)
fmt.Printf("Tokens:  \t%v\n", tokens)

Sentence:	The book is on the table.
Tokens:  	[101 1996 2338 2003 2006 1996 2795 1012 102 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


---

## HuggingFace Datasets

We are going to use the [HuggingFaceFW/fineweb](https://huggingface.co/datasets/HuggingFaceFW/fineweb) as an example, download one and parse one of its `.parquet` files.

In [46]:
import "github.com/gomlx/go-huggingface/datasets"

%%
// Print dataset info: configurations, splits, sizes and features.
ds := datasets.New("HuggingFaceFW/fineweb").WithAuth(hfAuthToken)
// ds := datasets.New("microsoft/ms_marco").WithAuth(hfAuthToken)

// Print only the last lines of the dataset description (FineWeb has too many versions):
// fmt.Println(ds.String())
lines := strings.Split(ds.String(), "\n")
const numLines = 18
if len(lines) > numLines {
    copy(lines[2:], lines[len(lines)-numLines+2:])
    lines = lines[:numLines]
}
fmt.Println(strings.Join(lines, "\n"))

Dataset ID: HuggingFaceFW/fineweb

Config: default
  Features: date, dump, file_path, id, language, language_score, text, token_count, url
  Splits: train (25.9G records, 85.7 TiB, 675 files)

Config: sample-100BT
  Features: date, dump, file_path, id, language, language_score, text, token_count, url
  Splits: train (147.6M records, 438.4 GiB)

Config: sample-10BT
  Features: date, dump, file_path, id, language, language_score, text, token_count, url
  Splits: train (14.9M records, 46.7 GiB)

Config: sample-350BT
  Features: date, dump, file_path, id, language, language_score, text, token_count, url
  Splits: train (518.5M records, 1.4 TiB)



### Parquet Structure 

We use the `github.com/gomlx/go-huggingface/cmd/generate_dataset_structs` to generate the Go 
structure for the Parquet files:

In [47]:
!go run ./cmd/generate_dataset_structs -dataset HuggingFaceFW/fineweb -config=default -split="train"

I0605 11:14:35.832793  278323 main.go:51] Found cached parquet file: "/home/janpf/.cache/huggingface/hub/datasets--HuggingFaceFW--fineweb/parquet/default/train/000_00000.parquet"
I0605 11:14:35.832814  278323 main.go:109] Generating Go struct from local parquet file "/home/janpf/.cache/huggingface/hub/datasets--HuggingFaceFW--fineweb/parquet/default/train/000_00000.parquet" with root struct "FinewebRecord"...


type FinewebRecord struct {
	Text string `json:"text" parquet:"text"`
	ID string `json:"id" parquet:"id"`
	Dump string `json:"dump" parquet:"dump"`
	URL string `json:"url" parquet:"url"`
	Date string `json:"date" parquet:"date"`
	FilePath string `json:"file_path" parquet:"file_path"`
	Language string `json:"language" parquet:"language"`
	LanguageScore float64 `json:"language_score" parquet:"language_score"`
	TokenCount int64 `json:"token_count" parquet:"token_count"`
}



### Iterating (Reading) Over Parquet Files

In [67]:
var (
    FineWebID = "HuggingFaceFW/fineweb"
    FineWebConfig = "sample-10BT"
    FineWebSplit = "train"
)

// FinewwebRecord generated with github.com/gomlx/go-huggingface/cmd/generate_dataset_structs
type FinewebRecord struct {
	Text string `json:"text" parquet:"text"`
	ID string `json:"id" parquet:"id"`
	Dump string `json:"dump" parquet:"dump"`
	URL string `json:"url" parquet:"url"`
	Date string `json:"date" parquet:"date"`
	FilePath string `json:"file_path" parquet:"file_path"`
	Language string `json:"language" parquet:"language"`
	LanguageScore float64 `json:"language_score" parquet:"language_score"`
	TokenCount int64 `json:"token_count" parquet:"token_count"`
}

// TrimString returns s trimmed to at most maxLength runes. If trimmed it appends "…" at the end.
func TrimString(s string, maxLength int) string {
    if utf8.RuneCountInString(s) <= maxLength {
        return s
    }
    runes := []rune(s)
    return string(runes[:maxLength-1]) + "…"
}

%%
ds := datasets.New(FineWebID)
ds.Verbosity = 2
count := 0
const limit = 10
for row, err := range datasets.IterParquetFromDataset[FinewebRecord](ds, FineWebConfig, FineWebSplit) {
    if err != nil { panic(err) }
	fmt.Printf("Record #%02d:\tScore=%.3f Text=%q, URL=[%s]\n", count+1, row.LanguageScore, TrimString(row.Text, 50), TrimString(row.URL, 40))
    count++
    if count >= limit { break }
}
fmt.Printf("%d records read!", count)

Record #01:	Score=0.823 Text="|Viewing Single Post From: Spoilers for the Week …", URL=[http://daytimeroyaltyonline.com/single/…]
Record #02:	Score=0.974 Text="*sigh* Fundamentalist community, let me pass on s…", URL=[http://endogenousretrovirus.blogspot.co…]
Record #03:	Score=0.873 Text="A novel two-step immunotherapy approach has shown…", URL=[http://news.cancerconnect.com/]
Record #04:	Score=0.932 Text="Free the Cans! Working Together to Reduce Waste\nI…", URL=[http://sharingsolution.com/2009/05/23/f…]
Record #05:	Score=0.955 Text="ORLANDO, Fla. — While the Rapid Recall Exchange, …", URL=[http://supermarketnews.com/food-safety/…]
Record #06:	Score=0.954 Text="September 28, 2010\n2010 Season - Bowman pulls dow…", URL=[http://www.augustana.edu/x22236.xml]
Record #07:	Score=0.967 Text="Kraft Foods has taken the Cadbury chocolate brand…", URL=[http://www.fdin.org.uk/2012/01/kraft-la…]
Record #08:	Score=0.874 Text="You must be a registered member to view this page…", URL=[http://www.goli

---

## HuggingFace ONNX models

The example below reads the `.onnx` model using a repo created with the package `hub`, creates a `tokenizer`,
uses the `bucket` to package a list of sentences into a padded batch, converts the model to a GoMLX model
and then executes it on the batch.


In [84]:
import (
    "github.com/gomlx/compute"
    "github.com/gomlx/go-huggingface/tokenizers/bucket"
    onnxparser "github.com/gomlx/onnx-gomlx/onnx/parser"
    "github.com/gomlx/gomlx/core/graph"
    "github.com/gomlx/gomlx/ml/model"

    // Default backends.
    _ "github.com/gomlx/gomlx/backends/default"
)

%%
// Get ONNX model.
repo := hub.New("sentence-transformers/all-MiniLM-L6-v2").WithAuth(hfAuthToken)
onnxFilePath, err := repo.DownloadFile("onnx/model.onnx")
if err != nil { panic(err) }
onnxModel, err := onnxparser.FromFile(onnxFilePath)
if err != nil { panic(err) }

// Convert ONNX variables to a GoMLX store:
store := model.NewStore()
err = onnxModel.VariablesToScope(store.RootScope())
if err != nil { panic(err) }

// Tokenize sentences.
tokenizer := must.M1(tokenizers.New(repo))
sentences := []string{
    "This is an example sentence", 
    "Each sentence is converted"}
batchSize := len(sentences)
sentencesChan := make(chan bucket.SentenceRef, batchSize)
bucketChan := make(chan bucket.Bucket, 1)
bucketizer := bucket.New(tokenizer).ByTwoBitBucket(batchSize, 8)
var wg sync.WaitGroup
wg.Go(func() { bucketizer.Run(sentencesChan, bucketChan) })
wg.Go(func() { 
    for i, s := range sentences { sentencesChan <- bucket.SentenceRef{s, i} }
    close(sentencesChan)
})

// Create GoMLX model, and its executor:
miniLMExec := model.MustNewExec1(
    compute.MustNew(), store, 
    func (scope *model.Scope, tokenIDs *graph.Node) *graph.Node {
        tokenIDs = graph.Reshape(tokenIDs, batchSize, -1)
        mask := graph.LogicalNot(graph.IsZero(tokenIDs))
        return onnxModel.CallGraph(scope, tokenIDs.Graph(), map[string]*graph.Node{
            "input_ids": tokenIDs,
            "attention_mask": graph.ConvertDType(mask, dtypes.Int64),
            "token_type_ids": graph.ZerosLike(tokenIDs)})[0]
    })

// Loop over batches:
for bucket := range bucketChan {
    tokenIDs := bucket.Batch
    embeddings := miniLMExec.MustCall(tokenIDs)
    fmt.Printf("Tokens: \t%v\n", tokenIDs)
    fmt.Printf("Embeddings:\t%s\n", embeddings)
}

Tokens: 	[101 2023 2003 2019 2742 6251 102 0 101 2169 6251 2003 4991 102 0 0]
Embeddings:	[2][8][384]float32{
 {{0.03652, -0.01617, 0.1683, ..., 0.05541, -0.1644, -0.2968},
  {0.7242, 0.6394, 0.189, ..., 0.5943, 0.6209, 0.4898},
  {0.006568, 0.02115, 0.04448, ..., 0.3471, 1.318, -0.1673},
  ...,
  {0.5212, 0.6566, 0.561, ..., -0.03989, 0.04128, -1.404},
  {1.083, 0.714, 0.3987, ..., -0.2289, 0.3248, -1.031},
  {-0.1745, 0.1791, 0.5735, ..., 0.1578, 0.002306, -0.4539}},
 {{0.2801, 0.1163, -0.04202, ..., 0.271, -0.1684, -0.2962},
  {0.8735, 0.4541, -0.1089, ..., 0.1362, 0.4584, -0.2045},
  {0.475, 0.5726, 0.6299, ..., 0.6521, 0.5611, -1.327},
  ...,
  {0.4114, 1.094, 0.2384, ..., 0.8982, 0.3684, -0.7336},
  {0.1354, 0.5587, 0.2699, ..., 0.5424, 0.47, -0.5306},
  {0.2323, 0.2985, 0.1732, ..., 0.4245, 0.07187, -0.3455}}}
